# 01 · Ingesta y exploración de PhysioNet 2019 — SepsisWatch Z

**Objetivo:** juntar los 40,336 archivos `.psv` (un paciente por archivo) en **una sola tabla**, y entender los datos antes de modelar.

**Entrada:** `~/shared/sepsis/data/raw/training_setA|B/*.psv` (descargados con `data/download_physionet.sh`)
**Salida:** `~/shared/sepsis/data/processed/sepsis_all.pkl` (pandas, `float32`)

Todo se ejecuta **dentro de la VM IBM LinuxONE (z17)**; ningún dato sale de la máquina.

## 1. Entorno: confirmar que corremos en IBM Z

In [ ]:
import os, platform, time, glob
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

with open("/proc/cpuinfo") as f:
    flags = next(line for line in f if line.startswith("features")).split()

print("Arquitectura :", platform.machine())
print("CPUs         :", os.cpu_count())
print("Acelerador IA (nnpa) en el chip:", "nnpa" in flags)
print("Vectores SIMD (vxe2)          :", "vxe2" in flags)

## 2. Rutas y columnas

Cada archivo tiene **41 columnas** en este orden: 8 signos vitales, 26 laboratorios, 6 datos del paciente y la etiqueta `SepsisLabel`.
El dataset **no trae** ID de paciente ni hospital: el ID está en el **nombre del archivo** y el hospital en la **carpeta**.

In [ ]:
BASE = os.path.expanduser("~/shared/sepsis/data")
RAW = os.path.join(BASE, "raw")
OUT_DIR = os.path.join(BASE, "processed")
OUT_FILE = os.path.join(OUT_DIR, "sepsis_all.pkl")
FIG_DIR = "../docs/figuras"   # figuras para el pitch (dentro del repo)

VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = ["BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
        "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
        "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium",
        "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC",
        "Fibrinogen", "Platelets"]
DEMOG = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
COLUMNS = VITALS + LABS + DEMOG + ["SepsisLabel"]
print(len(COLUMNS), "columnas")

## 3. Leer los 40,336 pacientes en paralelo

`read_patient` lee **un** archivo y le agrega `patient_id`, `hospital` y `hour`.
Con `Pool(os.cpu_count())` se leen **en paralelo usando los 3 CPUs del z17**, en lotes de 500 archivos.
Usamos `float32` (mitad de memoria que `float64`), que además es el formato con el que trabaja el acelerador NNPA.

In [ ]:
def read_patient(path):
    df = pd.read_csv(path, sep="|", dtype=np.float32)
    if list(df.columns) != COLUMNS:
        raise ValueError(f"Columnas inesperadas en {path}")
    df.insert(0, "patient_id", os.path.basename(path)[:-4])        # p000001.psv -> p000001
    df.insert(1, "hospital", "A" if "training_setA" in path else "B")
    df.insert(2, "hour", np.arange(len(df), dtype=np.int16))
    return df

files = sorted(glob.glob(os.path.join(RAW, "training_set*", "*.psv")))
print(f"Archivos encontrados: {len(files):,}")

t0 = time.time()
with Pool(os.cpu_count()) as pool:
    frames = pool.map(read_patient, files, chunksize=500)
t_read = time.time() - t0
print(f"Lectura en paralelo: {t_read:.1f} s en {platform.machine()} con {os.cpu_count()} CPUs")

## 4. Unir todo en una tabla y guardarla

In [ ]:
t1 = time.time()
df = pd.concat(frames, ignore_index=True)
del frames
df["patient_id"] = df["patient_id"].astype("category")   # cada ID se guarda una vez
df["hospital"] = df["hospital"].astype("category")
df["SepsisLabel"] = df["SepsisLabel"].astype(np.int8)

os.makedirs(OUT_DIR, exist_ok=True)
df.to_pickle(OUT_FILE)
t_total = t_read + (time.time() - t1)

print(f"Filas (horas-paciente): {len(df):,} | Columnas: {df.shape[1]}")
print(f"Memoria en RAM: {df.memory_usage(deep=True).sum() / 1024**2:,.0f} MB")
print(f"Guardado en: {OUT_FILE} ({os.path.getsize(OUT_FILE) / 1024**2:,.0f} MB)")
print(f"Tiempo total de ingesta: {t_total:.1f} s en {platform.machine()}")
df.head()

## 5. Resumen por hospital (cifras para el pitch)

Un paciente "tiene sepsis" si en algún momento su `SepsisLabel` vale 1.

In [ ]:
per_patient = df.groupby("patient_id", observed=True).agg(
    hospital=("hospital", "first"),
    hours=("hour", "size"),
    sepsis=("SepsisLabel", "max"),
    age=("Age", "first"),
    gender=("Gender", "first"),
)
summary = per_patient.groupby("hospital", observed=True).agg(
    pacientes=("sepsis", "size"),
    con_sepsis=("sepsis", "sum"),
    horas_mediana=("hours", "median"),
    edad_mediana=("age", "median"),
)
summary["pct_sepsis"] = (100 * summary["con_sepsis"] / summary["pacientes"]).round(2)
summary.loc["Total"] = [summary["pacientes"].sum(), summary["con_sepsis"].sum(),
                        per_patient["hours"].median(), per_patient["age"].median(),
                        round(100 * per_patient["sepsis"].mean(), 2)]
summary

## 6. ¿Cuántos datos faltan?

En la UCI los **signos vitales** se miden casi cada hora, pero los **laboratorios** pocas veces al día.
Por eso la mayoría de celdas son `NaN`. Esto define cómo construiremos las *features* (arrastrar el último valor medido).

In [ ]:
missing = (df[VITALS + LABS].isna().mean() * 100).sort_values()
colors = ["#2a9d8f" if c in VITALS else "#e76f51" for c in missing.index]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(missing.index, missing.values, color=colors)
ax.set_xlabel("% de horas sin medición")
ax.set_title("Datos faltantes por variable (verde = signo vital, naranja = laboratorio)")
ax.axvline(50, ls="--", c="grey", lw=1)
plt.tight_layout()
os.makedirs(FIG_DIR, exist_ok=True)
plt.savefig(f"{FIG_DIR}/01_faltantes.png", dpi=120)
plt.show()
missing.round(1).to_frame("% faltante").T

## 7. Un paciente con sepsis, hora por hora

`SepsisLabel` pasa a **1 seis horas antes** del inicio clínico de la sepsis (definición Sepsis-3).
Elegimos un paciente con sepsis, estadía larga y buenos registros, y vemos cómo cambian sus signos vitales.

- **Zona roja:** horas con `SepsisLabel = 1` (lo que el modelo debe detectar).
- **Línea negra:** inicio clínico de la sepsis (primera hora con etiqueta + 6).

In [ ]:
first_label = df[df["SepsisLabel"] == 1].groupby("patient_id", observed=True)["hour"].min()
cand = per_patient.loc[first_label.index].assign(first_label=first_label)
cand = cand[(cand["first_label"] >= 24) & (cand["hours"] >= cand["first_label"] + 10)]
hr_cov = df[df["patient_id"].isin(cand.index)].groupby("patient_id", observed=True)["HR"].count()
pid = (hr_cov.reindex(cand.index) / cand["hours"]).idxmax()

p = df[df["patient_id"] == pid].set_index("hour")
fl = int(cand.loc[pid, "first_label"])
onset = fl + 6
print(f"Paciente {pid} | hospital {cand.loc[pid, 'hospital']} | edad {cand.loc[pid, 'age']:.0f} | "
      f"{len(p)} h en UCI | etiqueta desde hora {fl} | inicio clínico ~ hora {onset}")

panels = [("HR", "Frecuencia cardíaca (lpm)"), ("Resp", "Frec. respiratoria (rpm)"),
          ("Temp", "Temperatura (°C)"), ("MAP", "Presión arterial media (mmHg)"),
          ("O2Sat", "Saturación O2 (%)")]
fig, axes = plt.subplots(len(panels), 1, figsize=(11, 12), sharex=True)
for ax, (col, label) in zip(axes, panels):
    ax.plot(p.index, p[col].ffill(), c="lightgrey", lw=1)
    ax.plot(p.index, p[col], "o-", ms=3, lw=1.2, c="#264653")
    ax.axvspan(fl, p.index.max(), color="#e63946", alpha=0.12)
    ax.axvline(onset, c="black", ls="--", lw=1)
    ax.set_ylabel(label, fontsize=9)
axes[0].set_title(f"Paciente {pid}: signos vitales antes de la sepsis")
axes[-1].set_xlabel("Horas en UCI")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_paciente_sepsis.png", dpi=120)
plt.show()

## 8. Patrón promedio: ¿qué cambia en las horas previas a la sepsis?

Alineamos a **todos** los pacientes con sepsis según las horas que faltan para el inicio clínico, y comparamos con el promedio de los pacientes **sin** sepsis.
Si las curvas se separan **antes** del inicio, hay señal para predecir con anticipación.

In [ ]:
sep = df[df["patient_id"].isin(first_label.index)].copy()
sep["onset"] = sep["patient_id"].map(first_label + 6).astype(float)
sep["rel"] = sep["hour"] - sep["onset"]
window = sep[(sep["rel"] >= -24) & (sep["rel"] <= 6)]
trend = window.groupby("rel")[["HR", "Resp", "Temp", "MAP"]].mean()
baseline = df[~df["patient_id"].isin(first_label.index)][["HR", "Resp", "Temp", "MAP"]].mean()

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), trend.columns):
    ax.plot(trend.index, trend[col], c="#e63946", lw=2, label="Con sepsis")
    ax.axhline(baseline[col], c="#2a9d8f", ls="--", label="Sin sepsis (promedio)")
    ax.axvline(0, c="black", lw=1)
    ax.axvspan(-6, 0, color="#e63946", alpha=0.08)
    ax.set_title(col)
    ax.set_xlabel("Horas respecto al inicio de la sepsis")
axes[0, 0].legend()
fig.suptitle("Signos vitales promedio en las 24 h previas a la sepsis")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_tendencia_previa.png", dpi=120)
plt.show()
trend.loc[[-24, -12, -6, 0]].round(2)

## 9. Conclusiones de la exploración

- Los datos quedan en **una sola tabla** lista para `02_features`, generada en la VM z17 en segundos usando los 3 CPUs en paralelo.
- La sepsis es **poco frecuente** (pocos % de pacientes): hay que medir con **AUPRC, sensibilidad y falsas alarmas**, no con *accuracy*.
- Los **laboratorios faltan en la gran mayoría de las horas**: las features deben arrastrar el último valor y marcar si hubo medición.
- Revisar en la sección 8 si las tendencias de frecuencia cardíaca, respiratoria y temperatura **se separan antes del inicio**: eso indica que hay señal para alertar con anticipación.
- Las figuras se guardaron en `docs/figuras/` para el pitch.